# Basic classification with PIE

Predict a participant's `COHORT` from the PPMI study-data download, end to end, and finish with
an HTML classification report you can open in a browser.

What this covers:

1. the one-call version, `run_pipeline(...)`;
2. the same run stage by stage, so you can look at the table between steps;
3. the parts that are easy to get wrong: the leakage list, the participant-level split, and which
   report holds the number you are allowed to quote.

**No outputs are saved in this notebook, on purpose.** PPMI data is released under a data use
agreement, so a notebook in this repository must not carry cohort counts, participant IDs or any
value derived from the download. Every cell below is unexecuted; running it on your own copy of
`./PPMI` fills the outputs in on your machine. Keep it that way before you share the file: clear
all outputs, or export the report instead of the notebook.

Depth on anything here: [pipeline.md](../documentation/pipeline.md),
[classifier.md](../documentation/classifier.md), and the
[documentation index](../documentation/README.md).

## Before you start

- PIE installed in the tabular environment (Python >= 3.10): `pip install -r requirements.txt && pip install -e .`
  from the repository root. That pulls in [PIE-clean](https://github.com/MJFF-ResearchCommunity/PIE-clean),
  which does the loading and cleaning.
- The PPMI **study-data** download unzipped at `./PPMI` (`_Subject_Characteristics/`, `Medical_History/`,
  `Motor___MDS-UPDRS/`, `Non-motor_Assessments/`, `Biospecimen/`, ...). PIE ships no data; apply for
  access at [ppmi-info.org](https://www.ppmi-info.org/access-data-specimens/download-data).
- Run the notebook from the repository root, so that `./PPMI`, `config/` and `output/` resolve.
- Imaging is optional and not needed here. The last section shows how imaging-derived phenotypes
  join if you have them.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from pie.pipeline import (
    run_pipeline,
    run_data_reduction_step,
    run_feature_engineering_step,
    run_feature_selection_step,
    run_classification_step,      # = pie.classification_report.generate_report
)

DATA_DIR = "./PPMI"                              # study-data download, read only by stage 1
LEAKAGE_PATH = "config/leakage_features.txt"
TARGET = "COHORT"

# A subset that loads in a couple of minutes. Drop `modalities` (or pass None) for the five
# core modalities, which adds biospecimen -- see "Cost and runtimes" near the end.
MODALITIES = ["subject_characteristics", "medical_history",
              "motor_assessments", "non_motor_assessments"]

OUT_ONE_CALL = Path("output/walkthrough_one_call")
OUT_STAGES = Path("output/walkthrough_stages")
OUT_STAGES.mkdir(parents=True, exist_ok=True)    # run_pipeline does this itself; the stage functions do not

# The final report and the summary page call webbrowser.open. On a headless machine
# (or a remote kernel) this keeps it from trying.
os.environ.setdefault("BROWSER", "true")

assert Path(DATA_DIR).is_dir(), f"No PPMI study-data download at {DATA_DIR}"
assert Path(LEAKAGE_PATH).exists(), f"No leakage list at {LEAKAGE_PATH}"

## The leakage list comes first

PPMI is full of columns that restate the diagnosis instead of predicting it: the clinician's
diagnosis code, enrolment flags, anti-parkinsonian treatment, substudy flags only one cohort
receives, the genetic variants that define the genetic cohorts. Leave them in and the model scores
beautifully by reading the answer off the form.

`config/leakage_features.txt` is one column name per line, matched **exactly** against the prefixed
names stage 1 produces (`subject_characteristics_APPRDX`, not `APPRDX`). `config/constants.py` holds
the same names as the Python list `LEAKAGE_FEATURES`. The file is used twice: stage 3 drops those
columns before feature selection, and stage 4 excludes them again. A path that does not exist logs `Leakage features file not found` and the run continues with
nothing removed, so watch for that line when you point this at your own copy.

**The shipped list is an example for PD-versus-control on `COHORT`, and it is wrong for your
question the moment you change the target.** To rebuild it: start from the columns whose definition
mentions the thing you are predicting, add anything a clinician recorded *because* of it, and, if
you are predicting a future state, anything measured after the time point you predict from. Copy
the file, edit your copy, and point `leakage_features_path=` at it rather than editing the shipped
one. After a run, the feature selection report lists the columns it removed, and the top features
in the classification report are the second check: a feature that looks too good usually is.

In [ ]:
leakage = [line.strip() for line in open(LEAKAGE_PATH) if line.strip()]
print(len(leakage), "names in", LEAKAGE_PATH)

# A few of them, by kind. PATNO and EVENT_ID are in the list but are never dropped by it:
# the IDs and the target are handled explicitly.
print([n for n in leakage if any(k in n for k in ("APPRDX", "NEWDIAG", "PDTRTMNT", "ENRLGBA"))])

## Path A: one call

`run_pipeline` runs all five stages and writes everything into `output_dir`. This is the whole
walkthrough in one cell; the rest of the notebook takes the same run apart.

Settings worth knowing:

- `fs_method="fdr"`, `fs_param_value=0.05` keeps the features whose univariate association with the
  target survives a 5 % false-discovery-rate cut. `k_best`, `rfe`, `mrmr` and `relief` read
  `fs_param_value` as *the fraction of features to keep* instead.
- `n_models_to_compare=3` refits the three top-ranked models on the full training split. The
  comparison itself always runs the whole default model set, whatever this number is.
- `tune_best_model=False` skips the 20-draw random search. Turn it on once the untuned run looks
  sane; the untuned model is kept if tuning does not beat it in the same CV.
- `budget_time_minutes=20.0` is checked **before each model starts**. It cannot interrupt a model
  that is already fitting, so a slow gradient-boosting model can overrun the budget by its own
  full fitting time.

In [ ]:
run_pipeline(
    data_dir=DATA_DIR,
    output_dir=str(OUT_ONE_CALL),
    target_column=TARGET,
    leakage_features_path=LEAKAGE_PATH,    # required: no default here, unlike the CLI
    modalities=MODALITIES,
    fs_method="fdr",
    fs_param_value=0.05,
    n_models_to_compare=3,
    tune_best_model=False,
    generate_plots=True,
    budget_time_minutes=20.0,
)

### What landed in the run directory

| File | Written by | What it is |
|---|---|---|
| `final_reduced_consolidated_data.csv`, `data_reduction_report.html` | 1 reduction | One row per `PATNO`/`EVENT_ID`, columns prefixed by source table; the report says what was dropped and why |
| `final_engineered_dataset.csv`, `feature_engineering_report.html` | 2 engineering | One-hot encoded and standardised |
| `selected_train_data.csv`, `selected_test_data.csv`, `feature_selection_report.html` | 3 split + selection | The two sides of the participant-level split, with the selected features, `PATNO` and the target |
| `classification/classification_report.html` | 4 classification | **The one to open.** Leaderboard, best model, plots, top-20 features, held-out test metrics |
| `classification/final_classifier_model.pkl` | 4 | joblib dict; reload with `Classifier().load_model(path)` |
| `classification/plots/` | 4 | Confusion matrix, SHAP, PCA, t-SNE, class distribution and endgame's own report; plus `feature.png` when the winning model exposes `feature_importances_` |
| `pipeline_report.html` | 5 summary | Links to every stage report from this invocation |

Start at `pipeline_report.html`, then `classification/classification_report.html`. In that report,
section 3 (leaderboard) and section 4 (best model) are **cross-validated scores on the training
split** -- that is how the model was chosen. Section 9, *Final Test Set Performance*, is the model
scored once on the held-out participants. Section 9 is the number you quote.

The command line does the same thing:

```bash
python pie/pipeline.py --data-dir ./PPMI --output-dir output/walkthrough_one_call \
    --target-column COHORT --modalities "subject_characteristics medical_history motor_assessments non_motor_assessments" \
    --fs-method fdr --fs-param 0.05 --n-models 3 --budget 20
```

In [ ]:
for p in sorted(OUT_ONE_CALL.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT_ONE_CALL))

## Path B: the same run, stage by stage

Each stage reads the previous stage's file out of the run directory and writes its own, which is
what makes `--skip-to` work. Calling the stage functions yourself buys you a look at the table in
between -- useful when a run produces a suspiciously good model, or no features at all.

Keep what you print at the shape and column level. `df.shape`, `df.columns[:10].tolist()` and
`nunique()` tell you whether a stage did its job without putting anyone's record in a notebook you
might share. For the class mix, print `value_counts(normalize=True)`: proportions answer "is this
badly imbalanced?" without publishing cohort counts from your download.

### Stage 1: load and reduce

`DataLoader.load` reads the download modality by modality; `DataReducer` then drops near-empty and
constant columns, merges the tables on `PATNO`/`EVENT_ID` and consolidates the several COHORT
columns into one. Columns come out prefixed with their source
(`subject_characteristics_AGE_AT_VISIT`, `motor_assessments_NP3TOT`).

Because `target_column="COHORT"`, rows without one of the four valid cohorts are dropped here. For
any other target nothing is dropped over a label you are not modelling.

In [ ]:
info = run_data_reduction_step(
    DATA_DIR,
    output_csv_path=OUT_STAGES / "final_reduced_consolidated_data.csv",
    output_html_path=OUT_STAGES / "data_reduction_report.html",
    modalities=MODALITIES,
    target_column=TARGET,
)
print("tables loaded -> kept:", info["initial_tables"], "->", info["reduced_tables"])
print("memory MB before -> after:", round(info["initial_size_mb"], 1), "->", round(info["reduced_size_mb"], 1))
print("consolidated shape:", info["output_shape"])

In [ ]:
reduced = pd.read_csv(OUT_STAGES / "final_reduced_consolidated_data.csv")

print("shape:", reduced.shape)
print("participants:", reduced["PATNO"].nunique(), "| visits per participant:", round(len(reduced) / reduced["PATNO"].nunique(), 2))
print("first columns:", reduced.columns[:10].tolist())
print("classes:", reduced[TARGET].nunique())
print(reduced[TARGET].value_counts(normalize=True).round(3))   # proportions, not counts

### Stage 2: engineering

`FeatureEngineer` one-hot encodes text columns with at most 20 distinct values (levels under 1 %
pooled into `_OTHER_`) and standardises every numeric column. The target is passed as a protected
column, so it is never encoded or scaled.

Two things to know: wider text columns stay text and get dropped in stage 3, and the scaling here
is fitted on **all** rows. Stage 3 re-fits it on the training participants, which is what actually
keeps the test rows out of it.

In [ ]:
fe = run_feature_engineering_step(
    str(OUT_STAGES / "final_reduced_consolidated_data.csv"),
    output_csv_path=OUT_STAGES / "final_engineered_dataset.csv",
    output_html_path=OUT_STAGES / "feature_engineering_report.html",
    target_column=TARGET,
)
print(fe["input_shape"], "->", fe["output_shape"], "| newly engineered:", fe["new_features"])

engineered = pd.read_csv(OUT_STAGES / "final_engineered_dataset.csv")
new_cols = [c for c in engineered.columns if c not in set(reduced.columns)]
print("examples of one-hot columns:", new_cols[:8])

### Stage 3: split, then select

The order matters, and it is the reason this stage does both.

1. Drop the leakage columns (exact match; the target, `PATNO` and `EVENT_ID` are never dropped by
   the list).
2. Drop rows with no target; `check_classification_target` raises `ValueError` if the target is
   continuous -- PIE classifies, it does not regress, so bin or map such a target first.
3. **Split by participant.** `split_train_test(y, groups=PATNO, test_size=0.2, random_state=42)`
   takes the first fold of a `StratifiedGroupKFold(n_splits=5)`: roughly 20 % of *participants*,
   with the class mix kept as close as whole participants allow. PPMI has several visits per
   person, and two visits of the same person are nearly the same row -- a plain row-level split
   would let the model score by recognising the person, and inflate every metric. The cell below
   checks the guarantee directly.
4. Re-standardise the continuous features on the training rows only, then fill `NaN` with 0, which
   is now the training mean. Feature selection is fitted on the training rows and applied to both.

The two CSVs hold the selected features, `PATNO` (so stage 4 can group its CV folds) and the target
with its original labels.

In [ ]:
fs = run_feature_selection_step(
    str(OUT_STAGES / "final_engineered_dataset.csv"),
    train_csv_path=OUT_STAGES / "selected_train_data.csv",
    test_csv_path=OUT_STAGES / "selected_test_data.csv",
    output_html_path=OUT_STAGES / "feature_selection_report.html",
    target_column=TARGET,
    fs_method="fdr",
    fs_param_value=0.05,
    leakage_features_path=LEAKAGE_PATH,
)
print("features:", fs["initial_features"], "->", fs["final_features"])
print("train / test shape:", fs["train_shape"], "/", fs["test_shape"])

train = pd.read_csv(OUT_STAGES / "selected_train_data.csv")
test = pd.read_csv(OUT_STAGES / "selected_test_data.csv")

print("participants on both sides:", len(set(train["PATNO"]) & set(test["PATNO"])))   # must be 0
print("leakage columns that survived:", [c for c in train.columns if c in leakage and c not in ("PATNO", TARGET)])
print("selected columns (first 10):", train.columns[:10].tolist())
print("train class mix:", train[TARGET].value_counts(normalize=True).round(3).to_dict())

### Stage 4: compare, score, report

`run_classification_step` is `pie.classification_report.generate_report` under its pipeline name.
Given a pre-split pair it uses the split as handed to it, cross-validates the model catalog on the
training side with folds grouped by `PATNO`, refits the top `n_models_to_compare`, optionally tunes,
then scores the final model once on the held-out participants.

`use_feature_selection=False` because stage 3 already selected. `exclude_features=leakage` is the
second application of the list -- harmless here, and the safety net when you feed this function a
CSV that never went through stage 3. It returns `(classifier, best_model, report_data)`;
`report_data` is what the HTML report is rendered from.

In [ ]:
classifier, best_model, report_data = run_classification_step(
    train_csv_path=str(OUT_STAGES / "selected_train_data.csv"),
    test_csv_path=str(OUT_STAGES / "selected_test_data.csv"),
    use_feature_selection=False,
    target_column=TARGET,
    exclude_features=leakage,
    output_dir=str(OUT_STAGES / "classification"),
    n_models_to_compare=3,
    tune_best_model=False,
    generate_plots=True,
    budget_time_minutes=20.0,
)

print("best model:", type(best_model).__name__)
print("report sections available:", sorted(report_data))        # keys only
print("open:", (OUT_STAGES / "classification" / "classification_report.html").resolve())

Open `output/walkthrough_stages/classification/classification_report.html`. There is no
`pipeline_report.html` in this directory: that summary page is written by stage 5, which only runs
when you call `run_pipeline`. Everything else is identical to Path A.

`report_data["test_metrics"]` holds the held-out numbers, `report_data["leaderboard"]` the
cross-validated comparison. Two different things; see
[classifier.md](../documentation/classifier.md#what-the-numbers-mean) before quoting either.

## Cost and runtimes

`modalities=` takes any of the five core modalities (`subject_characteristics`, `medical_history`,
`motor_assessments`, `non_motor_assessments`, `biospecimen`) plus PIE-clean's extended folders
(`study_enrollment`, `imaging`, `ppmi_online`, `remote_screening`, `found`, `roche_app`). Unknown
names are dropped with a warning. `None` means the five core ones -- **including biospecimen**.

Biospecimen is the expensive one by a wide margin: the proteomics projects are tens of thousands of
columns and dominate both the memory and the wall clock of stage 1, and they push stage 3 into
selecting from a very wide matrix. Start with the clinical subset used above, confirm the run works
end to end, then add `"biospecimen"` once you actually want assay features.

Rough expectations on a normal workstation, clinical modalities only:

| Stage | Order of magnitude |
|---|---|
| 1 reduction | minutes -- dominated by reading and cleaning the CSVs; add a lot with biospecimen |
| 2 engineering | about a minute |
| 3 split + selection | seconds to a minute |
| 4 classification | roughly your `budget_time_minutes`, plus overrun |

Stage 4 is the one you control. `budget_time_minutes` is checked before each model starts, so the
real cost is the budget plus however long the model that started last takes to finish; with wide
PPMI frames a gradient-boosting model can add several minutes on its own. `tune_best_model=True`
adds 20 more fits of the winner. If a run dies, restart it with `skip_to_step=` (`"reduction"`,
`"engineering"`, `"selection"`, `"classification"`) and it picks up from the files already in the
run directory.

## Adding imaging features

If you have run the imaging layer, `pie/imaging/run.py` leaves a `fastsurfer_idps.csv` in its
derived directory: one row per processed session, keyed by `PATNO` and `EVENT_ID`. Pass it as
`imaging_features=` and stage 1 joins it to the tabular modalities on those two columns -- the
numeric columns become features named `imaging_<column>` (or `imaging_idps_<column>` if you also
load the `imaging` modality, so the two do not collide). Text columns, `IMAGEID` and `SCAN_DATE`
are dropped, and repeated scans at one visit collapse to the first non-null value.

The one requirement is that `EVENT_ID` uses clinical visit codes: a scan whose `EVENT_ID` matches
no clinical visit becomes a row of its own, with no cohort, and is then dropped by the COHORT
filter. See [imaging.md](../documentation/imaging.md).

The cell below is a template -- it does nothing until you flip the flag.

In [ ]:
RUN_IMAGING_JOIN = False                                          # set True once you have the table
IDP_CSV = Path("Imaging/derived/fastsurfer_idps.csv")             # wherever you pointed --derived

if RUN_IMAGING_JOIN and IDP_CSV.exists():
    run_pipeline(
        data_dir=DATA_DIR,
        output_dir="output/walkthrough_with_imaging",
        target_column=TARGET,
        leakage_features_path=LEAKAGE_PATH,
        modalities=MODALITIES,
        imaging_features=str(IDP_CSV),                            # CLI: --imaging-features
        n_models_to_compare=3,
        budget_time_minutes=20.0,
    )
else:
    print("skipped -", IDP_CSV, "exists:", IDP_CSV.exists())

## Where to go next

- **Predict something else.** `target_column=` takes any column that holds class labels, text or
  integer codes (`"subject_characteristics_RBD"`, a binned score). A continuous column raises
  `ValueError`. Rebuild the leakage list for the new target before you believe any result: the
  shipped one only protects `COHORT`.
- **Change the selector.** `fs_method=` accepts `k_best`, `fdr`, `select_from_model`, `rfe` and,
  with endgame installed, `boruta`, `shap`, `mrmr`, `relief` and more --
  [feature_selector.md](../documentation/feature_selector.md).
- **Tighten the evaluation.** `pie.experiment` has cohort rules for PPMI's encoding traps, nested
  model selection that never touches its test partition, and provenance manifests --
  [experiment.md](../documentation/experiment.md).
- **Test a hypothesis instead of fitting a model.** [stats.md](../documentation/stats.md) has a
  "which test do I use?" table.
- **Full reference.** [pipeline.md](../documentation/pipeline.md) documents every flag, stage and
  output file; [classifier.md](../documentation/classifier.md) documents the classification layer;
  the [documentation index](../documentation/README.md) maps the rest.

Before sharing this notebook: **Kernel -> Restart & Clear Output**. The data use agreement applies
to anything computed from the download, cell outputs included.